In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# בניית Circuit

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלה או בחדשות יותר.

```
qiskit[all]~=2.3.0
```
</details>
דף זה בוחן לעומק את המחלקה [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) ב-Qiskit SDK, כולל מספר שיטות מתקדמות יותר שבהן תוכל להשתמש כדי ליצור Circuit קוונטיים.
## מהו Circuit קוונטי?
Circuit קוונטי פשוט הוא אוסף של Qubit ורשימת הוראות הפועלות עליהם. להמחשה, התא הבא יוצר Circuit חדש עם שני Qubit חדשים, ולאחר מכן מציג את המאפיין [`qubits`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) של ה-Circuit, שהוא רשימה של [`Qubit`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Qubit) בסדר מהביט הפחות משמעותי $q_0$ עד הביט המשמעותי ביותר $q_n$.

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Multiple `QuantumRegister` and `ClassicalRegister` objects can be combined to create a circuit. Every [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) and [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) can also be named.

In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

ניתן לשלב מספר אובייקטים מסוג `QuantumRegister` ו-`ClassicalRegister` כדי ליצור Circuit. לכל [`QuantumRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.QuantumRegister) ו-[`ClassicalRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) ניתן גם לתת שם.

In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Adding an instruction to the circuit appends the instruction to the circuit's [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) attribute. The following cell output shows `data` is a list of [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objects, each of which has an `operation` attribute, and a `qubits` attribute.

In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

תוכל למצוא את האינדקס והרגיסטר של Qubit על-ידי שימוש בשיטה [`find_bit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) של ה-Circuit ובמאפייניה.

In [5]:
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Circuit instruction objects can contain "definition" circuits that describe the instruction in terms of more fundamental instructions. For example, the [X-gate](/docs/api/qiskit/qiskit.circuit.library.XGate) is defined as a specific case of the [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate), a more general single-qubit gate.

In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

הוספת הוראה ל-Circuit מוסיפה את ההוראה למאפיין [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#data) של ה-Circuit. פלט התא הבא מראה כי `data` הוא רשימה של אובייקטים מסוג [`CircuitInstruction`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.CircuitInstruction), לכל אחד מהם יש מאפיין `operation` ומאפיין `qubits`.

In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

To combine two circuits, use the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method. This accepts another [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and an optional list of qubit mappings.

<Admonition type="note">
    The [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method returns a new circuit and does **not** mutate either circuit it acts on. To mutate the circuit on which you're calling the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method, use the argument `inplace=True`.
</Admonition>

In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

הדרך הקלה ביותר להציג מידע זה היא באמצעות שיטת [`draw`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#draw), שמחזירה המחשה ויזואלית של Circuit. ראה [המחשת Circuit](./visualize-circuits) לדרכים שונות להצגת Circuit קוונטיים.

In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg)

אובייקטים של הוראות Circuit יכולים להכיל Circuit "הגדרה" המתארים את ההוראה במונחים של הוראות בסיסיות יותר. לדוגמה, ה-[X-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.XGate) מוגדר כמקרה מיוחד של ה-[U3-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.U3Gate), Gate כללי יותר לQubit בודד.

In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg)

הוראות ו-Circuit דומים בכך ששניהם מתארים פעולות על ביטים ו-Qubit, אך יש להם מטרות שונות:

- הוראות נחשבות כקבועות, ושיטותיהן יחזירו בדרך כלל הוראות חדשות (מבלי לשנות את האובייקט המקורי).
- Circuit מיועדים לבנייה על פני שורות קוד רבות, ושיטות [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) לרוב משנות את האובייקט הקיים.
### מהי עומק Circuit?
ה-[depth()](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) של Circuit קוונטי הוא מדד למספר ה"שכבות" של Gate קוונטיים, המבוצעות במקביל, הנדרשות להשלמת החישוב המוגדר על ידי ה-Circuit. מכיוון שמימוש Gate קוונטיים לוקח זמן, העומק של Circuit תואם בקירוב לכמות הזמן שלוקח למחשב הקוונטי לבצע את ה-Circuit. לפיכך, עומק ה-Circuit הוא אחד הכמויות החשובות המשמשות למדידת האפשרות להריץ Circuit קוונטי על מכשיר.

שאר הדף ממחיש כיצד לתפעל Circuit קוונטיים.
## בניית Circuit
שיטות כגון [`QuantumCircuit.h`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#h) ו-[`QuantumCircuit.cx`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#cx) מוסיפות הוראות ספציפיות ל-Circuit. כדי להוסיף הוראות ל-Circuit באופן כללי יותר, השתמש בשיטה [`append`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#append). שיטה זו מקבלת הוראה ורשימת Qubit עליהם יש להחיל את ההוראה. ראה את [תיעוד ה-API של ספריית Circuit](https://docs.quantum.ibm.com/api/qiskit/circuit_library) לקבלת רשימת הוראות נתמכות.

In [11]:
qc_a.decompose().draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg)

כדי לשלב שני Circuit, השתמש בשיטה [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose). שיטה זו מקבלת [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) נוסף ורשימת מיפויי Qubit אופציונלית.

> **Note:** השיטה [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) מחזירה Circuit חדש ו**אינה** משנה את אף אחד מה-Circuit עליהם היא פועלת. כדי לשנות את ה-Circuit שעליו אתה קורא לשיטה [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose), השתמש בארגומנט `inplace=True`.

In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg)

כדי לראות מה קורה, תוכל להשתמש בשיטה [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) כדי לפרק כל הוראה להגדרתה.

> **Note:** השיטה [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) מחזירה Circuit חדש ו**אינה** משנה את ה-Circuit עליו היא פועלת.

In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg)

<span id="measure-qubits"></span>
## מדידת Qubit
מדידות משמשות לדגימת מצבי Qubit בודדים ולהעברת התוצאות לרגיסטר קלאסי. שים לב שאם אתה שולח Circuit ל-primitive מסוג [Sampler](./primitives#sampler), מדידות הן חובה. עם זאת, Circuit הנשלחים ל-primitive מסוג [Estimator](./primitives#estimator) אסור שיכילו מדידות.

ניתן למדוד Qubit בשלוש שיטות: [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure), [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) ו-[`measure_active`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). כדי ללמוד כיצד להמחיש תוצאות מדידה, ראה את דף [המחשת תוצאות](./visualize-results).

1. `QuantumCircuit.measure` : מודד כל Qubit בארגומנט הראשון על הביט הקלאסי הנתון כארגומנט השני. שיטה זו מאפשרת שליטה מלאה על מקום אחסון תוצאת המדידה.

2. `QuantumCircuit.measure_all` : אינה מקבלת ארגומנט וניתן להשתמש בה עבור Circuit קוונטיים ללא ביטים קלאסיים מוגדרים מראש. היא יוצרת חוטים קלאסיים ומאחסנת תוצאות מדידה בסדר. לדוגמה, מדידה של Qubit $q_i$ מאוחסנת ב-cbit $meas_i$). היא גם מוסיפה מחסום לפני המדידה.

3. `QuantumCircuit.measure_active` : דומה ל-`measure_all`, אך מודדת רק Qubit שיש להם פעולות.

In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

You can find a list of a circuit's undefined parameters in its `parameters` attribute.

In [17]:
qc.parameters

ParameterView([Parameter(angle)])

### Change a parameter's name

By default, parameter names for a parameterized circuit are prefixed by `x`- for example, `x[0]`. You can change the names after they are defined, as shown in the following example.

In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg)

## Circuit פרמטריים

אלגוריתמים קוונטיים רבים לטווח הקרוב כוללים הרצה של וריאציות רבות של Circuit קוונטי. מכיוון שבניית Circuit גדולים ואופטימיזציה שלהם עלולים להיות יקרים מבחינה חישובית, Qiskit תומך ב-Circuit **פרמטריים**. ל-Circuit אלה יש פרמטרים לא מוגדרים, וערכיהם אינם חייבים להיות מוגדרים עד ממש לפני הרצת ה-Circuit. זה מאפשר לך להוציא את בניית ה-Circuit ואופטימיזציה שלו מחוץ ללולאת התוכנית הראשית. התא הבא יוצר ומציג Circuit פרמטרי.